# contiguous-layout — ex2: fix the view-after-transpose error with .contiguous()

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `contiguous-layout`. Running the final beacon cell reports progress against the `PyTorch: Contiguous layout` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Contiguous layout` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`contiguous-layout`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "contiguous-layout"
DD_SUBTOPIC = "PyTorch: Contiguous layout"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Contiguous layout — quick refresher

A tensor is **contiguous** when its in-memory layout is row-major: the last axis has stride 1 and each earlier axis's stride equals the product of all sizes to its right. `x.is_contiguous()` reports the answer; `x.stride()` lets you check by hand.

**Why it matters.**
- `view()` requires contiguous input — call `.contiguous()` first if you've transposed/permuted/strided into a non-contiguous layout.
- `reshape()` will silently copy when needed; `view()` will not.
- Many low-level kernels (cuDNN convs, `as_strided`) read raw stride values — passing them a tensor whose strides you didn't expect is the single biggest source of off-by-`H*W` bugs in CNN-from-scratch code.

**Useful identities for a contiguous `(d0, d1, ..., dN)` tensor:**
- `stride(N) == 1`
- `stride(k) == d[k+1] * d[k+2] * ... * d[N]`

### Exercise 2 — fix the view-after-transpose error with .contiguous()

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Diagnose a `RuntimeError: view size is not compatible with input tensor's size and stride` after a transpose, and fix it by inserting `.contiguous()` before the `.view()` call.
> Keywords: view, transpose, contiguous-copy
> ```

**KCs targeted:** `view-requires-contiguous`, `contiguous-materialize`

Implement `ex2_flatten_after_transpose(x)`.

Input: `x` of shape `(B, H, W)`. You must:
1. Transpose the last two axes to get `(B, W, H)`.
2. Flatten the trailing two axes into one via `.view(B, W * H)`.
3. Return the `(B, W * H)` tensor.

**The catch.** A naive `x.transpose(-1, -2).view(B, W * H)` raises a `RuntimeError` because `transpose` returns a non-contiguous view and `view` refuses non-contiguous inputs. Fix it by calling `.contiguous()` between the two ops.

Output: `(B, W * H)` float tensor, row-major flattened from the transposed `(B, W, H)` layout.

In [ ]:
def ex2_flatten_after_transpose(x: Tensor) -> Tensor:
    """Transpose last two axes, then flatten via .view()."""
    raise NotImplementedError()


def _test_ex2():
    # Build a tensor where the value encodes its position so we can
    # verify the transpose actually happened (not just a reshape).
    x = t.arange(24, dtype=t.float32).reshape(2, 3, 4)
    out = ex2_flatten_after_transpose(x)
    assert out.shape == (2, 12), f'expected (2, 12), got {tuple(out.shape)}'
    assert out.dtype == t.float32, f'expected float32, got {out.dtype}'

    # Reference: do it the explicit safe way and compare.
    expected = x.transpose(-1, -2).contiguous().view(2, 12)
    assert t.equal(out, expected), (
        f'value mismatch:\nout={out}\nexpected={expected}'
    )

    # Confirm the result really came from the TRANSPOSED layout —
    # i.e. not a flat .view() of x itself (which would have produced
    # a different ordering, [0,1,2,3,4,...] instead of
    # [0,4,8,1,5,9,2,6,10,3,7,11] for batch 0).
    wrong = x.view(2, 12)
    assert not t.equal(out, wrong), (
        'output equals plain .view(B, W*H) — you skipped the transpose'
    )

    # Larger random case for shape robustness.
    rng = t.Generator().manual_seed(0)
    y = t.randn(5, 7, 11, generator=rng)
    out_y = ex2_flatten_after_transpose(y)
    assert out_y.shape == (5, 77)
    assert t.allclose(out_y, y.transpose(-1, -2).contiguous().view(5, 77))
    print('transpose + .contiguous() + .view fixed the RuntimeError')
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_flatten_after_transpose(x: Tensor) -> Tensor:
    B = x.shape[0]
    H, W = x.shape[1], x.shape[2]
    return x.transpose(-1, -2).contiguous().view(B, W * H)
```

**Why `transpose` breaks `view`.** `transpose` keeps the same storage but swaps two stride values. The new strides don't satisfy the contiguous formula, so `view` (which is a pure metadata re-interpretation) cannot re-label them as a flat 1-D buffer.

**Two valid fixes.**
- `x.transpose(-1, -2).contiguous().view(B, W * H)` — explicit, shows your intent.
- `x.transpose(-1, -2).reshape(B, W * H)` — `reshape` calls `.contiguous()` for you under the hood when it has to.

**Cost of `.contiguous()`.** It allocates and copies — `O(B * H * W)` memory. For very large activation maps this is the kind of hidden allocation that shows up in profilers as a 'mysterious' 2x memory blip.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()